# 03 — Mood-to-Genre Mapping

**Purpose:** Define how user-selected moods are converted into movie genre preferences.

**Why this matters:** The TMDB dataset contains no explicit mood field. Our system must define a mapping from moods to genres so that all three algorithms (collaborative filtering, content-based, hybrid) use the same mood framework.

**Approach:** Rule-based mapping justified by Mood Management Theory (Zillmann, 1988) and empirical studies on emotion-genre associations.

**Output:** A `mood_mapping.csv` file and mood-enriched datasets saved to `data/processed/`.

## 1. Theoretical Background

### Mood Management Theory (MMT)

The foundational framework for our mood-genre mapping is **Mood Management Theory** (Zillmann, 1988). MMT proposes that people select entertainment media to regulate their emotional state — specifically to **maintain positive moods** and **repair negative moods**.

Key predictions relevant to our system:
- People in **negative moods** (sad, stressed, bored) tend to seek **positive-affect** content (comedy, animation, family films)
- People in **positive moods** (happy, excited) tend to seek **arousing or congruent** content (action, adventure, romance)
- The goal is mood **optimisation**, not necessarily mood **congruence** (Zillmann, 2000)

### Empirical Support

- **Winoto & Tang (2010)** directly demonstrated that hierarchical mood states map to specific genre preferences in movie recommendations
- **Knobloch-Westerwick (2007)** found that individuals in negative moods preferentially select positive-affect genres
- **Greenwood (2010)** showed that mood shifts alter genre preferences even within the same individual
- **Qiao (2021)** found that perceived stress levels correlate with genre preferences

### Important Caveat

Koopman (2015) notes that some viewers deliberately choose sad or challenging content for eudaimonic (meaning-driven) reasons. Our system therefore offers mood-congruent options alongside mood-repair options.

## 2. Define Mood Categories

In [ ]:
# Define the six mood categories used in our system.
# These are the moods a user can select as input.
MOODS = ['Happy', 'Sad', 'Stressed', 'Excited', 'Romantic', 'Bored']

print(f'Mood categories: {MOODS}')
print(f'Total moods: {len(MOODS)}')

## 3. Mood-to-Genre Mapping

Each mood is mapped to a set of genres based on:
1. **Mood Management Theory** — people seek content that optimises their mood
2. **Empirical studies** — genre preferences under different emotional states
3. **Practical coverage** — ensuring each mood has enough movies to recommend from

### Mapping Rules

| Mood | Primary Genres | Rationale | Source |
|---|---|---|---|
| **Happy** | Comedy, Animation, Family, Music | Happy viewers seek light, enjoyable content that maintains their positive state | Zillmann (1988); Knobloch-Westerwick (2007) |
| **Sad** | Drama, Romance, Comedy | Sad viewers seek either mood-congruent catharsis (drama) or mood repair (comedy) | Zillmann (2000); Koopman (2015) |
| **Stressed** | Comedy, Animation, Family, Documentary | Stressed viewers need relaxation and escapism; light genres reduce tension | Winoto & Tang (2010); Qiao (2021) |
| **Excited** | Action, Adventure, Thriller, Science Fiction | Excited viewers seek high-arousal content that matches their energy | Greenwood (2010) |
| **Romantic** | Romance, Drama, Comedy | Romantic mood calls for love stories and emotional narratives | Winoto & Tang (2010) |
| **Bored** | Adventure, Action, Science Fiction, Mystery, Horror | Bored viewers need stimulating, unpredictable content to break monotony | Zillmann (1988); Knobloch-Westerwick (2007) |

In [ ]:
# Define the mood-to-genre mapping.
# Each mood maps to a list of genres that research suggests are preferred under that mood.
mood_to_genres = {
    'Happy': ['Comedy', 'Animation', 'Family', 'Music'],
    'Sad': ['Drama', 'Romance', 'Comedy'],
    'Stressed': ['Comedy', 'Animation', 'Family', 'Documentary'],
    'Excited': ['Action', 'Adventure', 'Thriller', 'Science Fiction'],
    'Romantic': ['Romance', 'Drama', 'Comedy'],
    'Bored': ['Adventure', 'Action', 'Science Fiction', 'Mystery', 'Horror'],
}

# Display the mapping
for mood, genres in mood_to_genres.items():
    print(f'{mood:10s} -> {genres}')

## 4. Validate Mapping Against Dataset

Before using this mapping, we must verify that the dataset actually contains enough movies in each mood's genres. If a mood maps to genres with very few movies, recommendations will be poor.

In [ ]:
import pandas as pd
import json
import os

In [ ]:
# Load the raw movies data for validation
movies_raw = pd.read_csv('../data/raw/tmdb_movie_dataset.csv')
ratings_raw = pd.read_csv('../data/raw/tmdb_movie_ratings.csv')

# Parse genres into lists
def parse_json_names(json_str):
    try:
        items = json.loads(json_str)
        return [item['name'] for item in items]
    except (json.JSONDecodeError, TypeError, KeyError):
        return []

movies_raw['genre_list'] = movies_raw['genres'].apply(parse_json_names)

# Keep only movies with ratings
valid_rids = set(ratings_raw['ratingId'])
movies = movies_raw[movies_raw['ratingId'].isin(valid_rids)].copy()

print(f'Valid movies (with ratings): {len(movies)}')

In [ ]:
# For each mood, count how many movies match its genres
print('=== Movie coverage per mood ===')
print(f'{"Mood":10s} {"Genres":<55s} {"Movies":>8s}')
print('-' * 75)

mood_coverage = {}
for mood, genres in mood_to_genres.items():
    # A movie matches a mood if it has ANY of the mood's genres
    mask = movies['genre_list'].apply(lambda g: any(genre in g for genre in genres))
    count = mask.sum()
    mood_coverage[mood] = count
    print(f'{mood:10s} {str(genres):<55s} {count:>8,}')

print()

# Check minimum coverage — if any mood has fewer than 100 movies, the mapping needs adjustment
min_coverage = min(mood_coverage.values())
min_mood = min(mood_coverage, key=mood_coverage.get)
if min_coverage < 100:
    print(f'WARNING: "{min_mood}" has only {min_coverage} movies. Consider adding more genres.')
else:
    print(f'All moods have sufficient coverage (minimum: {min_mood} = {min_coverage:,} movies)')

In [ ]:
# Show the rating count for each mood's movies
# This tells us how many user ratings exist for each mood category
print('=== Total ratings per mood ===')

# Build a ratingId -> genre_list lookup
rid_to_genres = movies.set_index('ratingId')['genre_list'].to_dict()

for mood, genres in mood_to_genres.items():
    # Find ratingIds whose genres overlap with this mood's genres
    matching_rids = [rid for rid, g in rid_to_genres.items()
                     if any(genre in g for genre in genres)]
    # Count ratings for those movies
    rating_count = ratings_raw[ratings_raw['ratingId'].isin(matching_rids)].shape[0]
    print(f'{mood:10s} -> {rating_count:>12,} ratings across {len(matching_rids):,} movies')

## 5. Visualise Coverage

In [ ]:
import matplotlib.pyplot as plt

# Bar chart of movie count per mood
fig, ax = plt.subplots(figsize=(10, 5))
moods = list(mood_coverage.keys())
counts = list(mood_coverage.values())
colors = ['#FFD700', '#6495ED', '#98FB98', '#FF6347', '#FF69B4', '#DDA0DD']

bars = ax.bar(moods, counts, color=colors, edgecolor='black')
ax.set_title('Number of Movies Available per Mood Category')
ax.set_xlabel('Mood')
ax.set_ylabel('Number of Movies')

# Add count labels on bars
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f'{count:,}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Save Mood Mapping

Save the mapping so all algorithms can load it.

In [ ]:
# Convert mood_to_genres to a DataFrame for easy CSV export
rows = []
for mood, genres in mood_to_genres.items():
    rows.append({
        'mood': mood,
        'genres': '|'.join(genres),
        'movie_count': mood_coverage[mood]
    })

mood_df = pd.DataFrame(rows)
os.makedirs('../data/processed', exist_ok=True)
mood_df.to_csv('../data/processed/mood_genre_mapping.csv', index=False)
print('Saved mood_genre_mapping.csv')
print()
print(mood_df.to_string(index=False))

## 7. Summary

### Mapping Design

The mood-to-genre mapping is based on:
- **Mood Management Theory** (Zillmann, 1988, 2000) — the theoretical foundation
- **Winoto & Tang (2010)** — empirical mood-genre associations in movie recommendations
- **Knobloch-Westerwick (2007)** — media selection under different emotional states
- **Koopman (2015)** — nuance that viewers sometimes seek mood-congruent content

### Design Decisions

1. **Six moods** selected for practical coverage and user intuitiveness
2. **Multiple genres per mood** to ensure sufficient movie coverage
3. **Overlapping genres** across moods (e.g., Comedy appears in Happy, Sad, and Stressed) — this is intentional, as comedy serves different mood-regulation functions (maintenance, repair, relaxation)
4. **Mood-repair as primary strategy** — negative moods (Sad, Stressed, Bored) map to positive-arousing genres, consistent with MMT
5. **Mood-congruent options included** — Sad also includes Drama for viewers seeking catharsis (Koopman, 2015)

### References

1. Zillmann, D. (1988). Mood management through communication choices. *American Behavioral Scientist*, 31(3), 327–340.

2. Zillmann, D. (2000). Mood management in the context of selective exposure theory. *Annals of the International Communication Association*, 23(1), 183–212.

3. Winoto, P., & Tang, T. Y. (2010). The role of user mood in movie recommendations. *Expert Systems with Applications*, 37(8), 6086–6092.

4. Knobloch-Westerwick, S. (2007). Gender differences in selective media use for mood management and mood adjustment. *Journal of Broadcasting & Electronic Media*, 51(2), 235–254.

5. Greenwood, D. N. (2010). Of sad men and dark comedies: Mood and gender effects on entertainment media preferences. *Mass Communication and Society*, 13(3), 230–249.

6. Koopman, E. M. E. (2015). Why do we read sad books? Eudaimonic motives and meta-emotions. *Poetics*, 51, 8–18.

7. Qiao, N. (2021). Does perceived stress of university students affected by preferences for movie genres? *Frontiers in Psychology*, 12, 700759.